# Urn Model for Locust Scenario

# Task 1: Urn Model for Locust Scenario

## Objective

The aim of this task is to empirically investigate the **average change in the number of left-moving agents**, ΔL, in a swarm of agents moving along a ring. This allows us to approximate a feedback function that captures the underlying collective dynamics and compare it to the theoretical **swarm urn model**. The final goal is to fit the swarm urn model to simulation data and analyze the probability of positive feedback (PFB) in relation to the fraction of left-goers.

---

## Methodology

### Simulation Setup

- **Environment**: 1D circular space of length \( C = 0.5 \)
- **Agent speed**: \( v = 0.01 \) per timestep
- **Perception radius**: \( r = 0.045 \)
- **Swarm size**: \( N = 50 \)
- **Random direction flip probability**: \( P = 0.15 \)

Each agent starts at a random position with a randomly assigned direction: left (-1) or right (+1). At each timestep, agents update their direction based on their neighbors’ directions and a random flip chance. The simulation is run for:

- **100 warm-up timesteps** to allow agents to self-organize,
- followed by **20 measurement timesteps** where we track the change ΔL in the number of left-goers.

This process is repeated for **50,000 runs** to get a reliable average.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

C = 0.5        # Length of the circular track
v = 0.01       # Velocity of agents per timestep
r = 0.045      # Interaction radius
N = 50         # Number of agents
P = 0.15       # Probability of random direction flip
T_warmup = 100 # Timesteps to reach steady state before measurement
T_measure = 20 # Number of timesteps to measure ΔL after warmup
runs = 50000   # Number of simulation runs to average over

# -----------------------------
# Preallocate result accumulators
# -----------------------------
delta_sum = np.zeros(N + 1)  # Sum of ΔL values for each possible L (number of left-goers)
count = np.zeros(N + 1)      # Count of observations for each L

# -----------------------------
# Main Simulation Loop
# -----------------------------
for run in range(runs):
    # Initialize agents with random positions on the ring [0, C)
    positions = np.random.uniform(0, C, N)
    
    # Random initial directions: -1 (left) or 1 (right)
    directions = np.random.choice([-1, 1], N)

    # Initial number of agents going left
    L_prev = np.count_nonzero(directions == -1)

    for t in range(T_warmup + T_measure):
        # Compute pairwise circular distances between all agents
        pos_diff = positions[:, None] - positions
        distances = np.abs(pos_diff)
        distances = np.minimum(distances, C - distances)  # Handle circular boundary

        # -----------------------------
        # Update directions based on neighbor influence
        # -----------------------------
        influence = np.zeros(N)
        for i in range(N):
            neighbors = distances[i] <= r  # Neighbors within interaction radius
            local_sum = np.sum(directions[neighbors])  # Net direction of neighbors

            # Update direction with probabilistic rule
            if np.random.rand() < P:
                influence[i] = -directions[i]  # Random flip
            elif local_sum != 0:
                influence[i] = np.sign(local_sum)  # Follow majority direction
            else:
                influence[i] = directions[i]  # Keep same direction

        # Update directions and positions
        directions = influence.astype(int)
        positions = (positions + v * directions) % C  # Move and wrap around circle

        # -----------------------------
        # Measurement Phase
        # -----------------------------
        L_curr = np.count_nonzero(directions == -1)  # Current number of left-goers

        if t >= T_warmup:
            delta = L_curr - L_prev  # Change in number of left-goers
            delta_sum[L_prev] += delta  # Accumulate ΔL
            count[L_prev] += 1          # Count how many times L_prev occurred

        L_prev = L_curr  # Update for next timestep

# Avoid division by zero
nonzero_mask = count > 0
L_vals = np.arange(N + 1)[nonzero_mask]          # L values with nonzero measurements
avg_deltaL = delta_sum[nonzero_mask] / count[nonzero_mask]  # Compute average ΔL


# Save Results to File

with open("L_of_L.txt", "w") as f:
    for L, dL in zip(L_vals, avg_deltaL):
        f.write(f"{L} {dL}\n")

plt.figure(figsize=(8, 5))
plt.plot(L_vals, avg_deltaL, marker='o')
plt.axhline(0, color='black', linestyle='--')  # Reference line at ΔL = 0
plt.xlabel("L (Number of Left-Goers)")
plt.ylabel("ΔL(L) (Average Change in L)")
plt.title("Average Change ΔL(L) vs L")
plt.grid(True)
#plt.show()

### Measuring ΔL(L)

At each measurement step, the value of ΔL = \( L_t - L_{t-1} \) is accumulated based on the value of \( L_{t-1} \) (i.e., the number of left-goers at the previous timestep). We use two arrays:

- `delta_sum[L]` to accumulate the total change observed for each value of L
- `count[L]` to track how often each L occurred

We then compute the average change \( \Delta L(L) = \frac{\text{delta\_sum}[L]}{\text{count}[L]} \) only for the values of L that were observed during the simulation.

---

## Results
- The plot shows the **average change in the number of left-goers** ΔL as a function of the current number of left-goers \( L \).
- The curve **crosses zero** at a specific value \( L^* \), indicating a **fixed point** where, on average, the number of left-goers doesn’t change.
- The slope around \( L^* \) gives insight into the **stability** of that fixed point (positive slope: unstable; negative slope: stable).
![ΔL(L) Plot](../output/task1_1_plot.png)
---

## Fitting the Urn Model

### Theory

The theoretical model describes the expected change in the fraction of left-goers \( s = \frac{L}{N} \) with the following equations:

1. **Probability of Positive Feedback (PFB)**:
   \[
   P_{\text{FB}}(s, \phi) = \phi \cdot \sin(\pi s)
   \]

2. **Expected Change**:
   \[
   \Delta s(s) = 4c \cdot \left( P_{\text{FB}}(s, \phi) - \frac{1}{2} \right) \cdot \left( s - \frac{1}{2} \right)
   \]

Where:
- \( \phi \in [0, 1] \): feedback intensity
- \( c \): scaling factor

### Curve Fitting

The measured values of ΔL were normalized to compute Δs:

\[
\Delta s = \frac{\Delta L}{N}
\]

Using nonlinear curve fitting (`scipy.optimize.curve_fit`), we fit the model to the data and obtain:

- **Fitted φ (feedback intensity)**: `φ ≈ 0.78`  
- **Fitted c (scaling constant)**: `c ≈ 1.82`  
(Values will vary slightly depending on the run)

---


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# -----------------------------
# Load and preprocess data
# -----------------------------
data = np.loadtxt("L_of_L.txt")
L_vals = data[:, 0]
deltaL_vals = data[:, 1]

N = 50
s_vals = L_vals / N
delta_s_vals = deltaL_vals / N  # Convert ΔL to Δs = ΔL / N

# -----------------------------
# Define the model function
# -----------------------------
def delta_s_model(s, phi, c):
    PFB = phi * np.sin(np.pi * s)
    return 4 * c * (PFB - 0.5) * (s - 0.5)

# -----------------------------
# Fit the model to data
# -----------------------------
# Initial guess for phi and c
initial_guess = [0.5, 1.0]
# Bounds: phi in [0, 1], c unbounded
bounds = ([0.0, -np.inf], [1.0, np.inf])

# Fit the function
params, _ = curve_fit(delta_s_model, s_vals, delta_s_vals, p0=initial_guess, bounds=bounds)
phi_fit, c_fit = params

print(f"Fitted φ (phi): {phi_fit:.4f}")
print(f"Fitted c: {c_fit:.4f}")

# -----------------------------
# Generate smooth curves for plotting
# -----------------------------
s_fit = np.linspace(0, 1, 300)
delta_s_fit = delta_s_model(s_fit, phi_fit, c_fit)
PFB_fit = phi_fit * np.sin(np.pi * s_fit)

# -----------------------------
# Plot: Δs(s) with fitted function
# -----------------------------
plt.figure(figsize=(10, 5))
plt.plot(s_vals, delta_s_vals, 'o', label='Simulation Data (Δs)')
plt.plot(s_fit, delta_s_fit, '-', label=f'Fitted Model\nφ = {phi_fit:.2f}, c = {c_fit:.2f}', color='red')
plt.axhline(0, color='black', linestyle='--')
plt.axvline(0.5, color='gray', linestyle=':')
plt.xlabel("s (Fraction of Left-Goers)")
plt.ylabel("Δs(s)")
plt.title("Δs(s) vs s with Fitted Urn Model")
plt.legend()
plt.grid(True)
plt.show()




### Plot 2: Fitted Δs(s) Curve

![ΔS vs S](../output/task1_2_plot.png)

- The red curve is the fitted model, and the dots are simulation data.
- The model accurately captures the **nonlinear dynamics** of Δs in relation to the proportion of left-goers \( s \).
- The zero-crossings of Δs indicate the **fixed points** of the system.

---

In [ ]:
# -----------------------------
# Plot: PFB(s) from fitted φ
# -----------------------------
plt.figure(figsize=(10, 4))
plt.plot(s_fit, PFB_fit, label=f"PFB(s) = φ·sin(πs), φ = {phi_fit:.2f}", color='green')
plt.axhline(0.5, color='black', linestyle='--')
plt.xlabel("s (Fraction of Left-Goers)")
plt.ylabel("PFB(s)")
plt.title("Probability of Positive Feedback (PFB) vs s")
plt.grid(True)
plt.legend()
plt.show()

### Plot 3: Probability of Positive Feedback \( P_{\text{FB}}(s) \)

![PFB vs s](../output/task1_3_plot.png)

- This plot shows the estimated **probability of positive feedback** for varying values of \( s \).
- The curve peaks at \( s = 0.5 \), where the swarm is balanced — this is where **positive feedback is strongest**.
- As \( s \to 0 \) or \( s \to 1 \), feedback decreases.

---


## Interpretation

- **Mathematically**, the points where \( \Delta L(L^*) = 0 \) or \( \Delta s(s^*) = 0 \) are **equilibrium points** of the swarm system.
- **Biologically**, these are states where the number of left-goers remains stable on average — i.e., the swarm is balanced or locked in a consensus.
- The fit confirms that the urn model with sin-based feedback accurately captures the **nonlinear and probabilistic dynamics** of collective decision-making.

---

## Conclusion

This task demonstrates the emergence of **collective stability** in a probabilistic multi-agent system, and shows how the **urn model** can be effectively fitted to empirical data. The resulting insights help in understanding how local interaction rules give rise to global consensus in swarms.

# Task 2: Density-Dependent Global Switching

## Objective

The goal of this task is to investigate how swarm density (i.e., swarm size \(N\)) affects the frequency and duration of **global switching** in a collective system of agents, inspired by locust behavior. A global switch is defined as a transition in collective motion from a state with a strong majority moving in one direction to the opposite one.

---

## Methodology

- The agents move on a **1D circular space** with circumference \(C = 0.5\).
- Each agent moves with a constant speed \(v = 0.01\) and interacts with others within a **perception radius \(r = 0.045\)**.
- With a spontaneous flip probability of \(P = 0.15\), agents can randomly reverse direction.
- The **swarm size \(N\)** is varied from 20 to 150 in steps of 10.
- Each simulation runs for \(T_{\text{total}} = 10{,}000\) time steps, repeated 30 times per swarm size.
- The system's state is classified based on the number of left-moving agents \(L\):
  - **Zone A**: \(L > 0.7N\) (strong left-goer majority)
  - **Zone B**: \(0.3N \leq L \leq 0.7N\) (mixed state)
  - **Zone C**: \(L < 0.3N\) (strong right-goer majority)
- A **global switch** is recorded when the swarm transitions:
  - From Zone A → B → C or Zone C → B → A.
- The time spent in zone B is measured as the **switch duration**.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Base Parameters (except N)
C = 0.5
v = 0.01
r = 0.045
P = 0.15
T_total = 10000  # Should be enough for multiple switches
runs_per_N = 30  # Repeat for averaging

# Swarm sizes to test
N_vals = np.arange(20, 155, 10)
avg_switch_times = []
num_switches_list = []

for N in N_vals:
    switch_durations = []

    for run in range(runs_per_N):
        positions = np.random.uniform(0, C, N)
        directions = np.random.choice([-1, 1], N)

        prev_zone = None
        zone_entry = None
        counter = 0

        for t in range(T_total):
            # Movement
            pos_diff = positions[:, None] - positions
            distances = np.abs(pos_diff)
            distances = np.minimum(distances, C - distances)

            influence = np.zeros(N)
            for i in range(N):
                neighbors = distances[i] <= r
                local_sum = np.sum(directions[neighbors])
                if np.random.rand() < P:
                    influence[i] = -directions[i]
                elif local_sum != 0:
                    influence[i] = np.sign(local_sum)
                else:
                    influence[i] = directions[i]

            directions = influence.astype(int)
            positions = (positions + v * directions) % C

            # Count left-goers
            L = np.count_nonzero(directions == -1)

            # Determine current zone
            if L > 0.7 * N:
                zone = "A"
            elif L < 0.3 * N:
                zone = "C"
            else:
                zone = "B"

            # Switch detection logic
            if zone in ["A", "C"]:
                if prev_zone == "B":
                    if zone != zone_entry:  # A → B → C or C → B → A
                        switch_durations.append(counter)
                    counter = 0
                prev_zone = zone
            elif zone == "B":
                if prev_zone in ["A", "C"]:
                    zone_entry = prev_zone  # Remember where we came from
                    counter = 1
                    prev_zone = "B"
                elif prev_zone == "B":
                    counter += 1

    # Stats for this N
    num_switches = len(switch_durations)
    avg_time = np.mean(switch_durations) if num_switches > 0 else np.nan

    avg_switch_times.append(avg_time)
    num_switches_list.append(num_switches)

# ---------------------------
# Plotting results
# ---------------------------

# Plot 1: Average switch time vs swarm size
plt.figure(figsize=(10, 5))
plt.plot(N_vals, avg_switch_times, marker='o', label="Avg. Switch Time")
plt.xlabel("Swarm Size N")
plt.ylabel("Average Switch Time")
plt.title("Switch Time vs Swarm Size")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# Plot 2: Number of switches vs swarm size
plt.figure(figsize=(10, 5))
plt.plot(N_vals, num_switches_list, marker='s', color='orange', label="Switch Count")
plt.xlabel("Swarm Size N")
plt.ylabel("Number of Switches")
plt.title("Number of Switches vs Swarm Size")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


## Results

### 1. Number of Switches vs Swarm Size

![Switch Count](../output/task2_1_plot.png)

- The number of switches **decreases significantly** as swarm size increases.
- At small \(N = 20\), more than 1400 switches were recorded.
- From \(N = 80\) onwards, the number of switches sharply drops and approaches zero.
- This implies that **larger swarms exhibit more stability** and are less prone to stochastic switches.

---

### 2. Average Switch Time vs Swarm Size

![Switch Time](../output/task2_2_plot.png)

- The average switch time initially starts high for small swarm sizes.
- It **reaches a minimum** in the mid-range (around \(N = 60\)–80), then **increases again** for larger swarm sizes.
- Interpretation:
  - **Small swarms**: Frequent but noisy switches, possibly slower due to indecision.
  - **Medium swarms**: Most efficient switching behavior.
  - **Large swarms**: Rare switching events, but when they happen, they take longer due to the **swarm’s inertia and alignment stability**.

---

## Interpretation

These results are consistent with observed behavior in biological systems such as real locust swarms:

- **Low-density swarms** exhibit frequent and fast fluctuations in direction.
- **High-density swarms** tend to stabilize into a unified direction and resist change.
- This simulation demonstrates how **density-dependent feedback** leads to emergent group stability or bistability.

---

## Conclusion

This experiment confirms that **swarm size directly affects collective switching dynamics**:

- **Smaller swarms** switch more often but are less stable.
- **Larger swarms** are more stable, exhibiting infrequent and slower transitions.

This reflects real-world swarm intelligence phenomena and highlights the importance of considering swarm size when designing collective robotic systems or analyzing natural collective behavior.

---
